## ????
??????? csv, ??????? Kaggle ??

Hints: https://stackoverflow.com/questions/16923281/pandas-writing-dataframe-to-csv-file

# [????]
- ?????????, ????????????? Kaggle ?

# [????]
- ???, ?????, ??????? csv ? (In[10]) (Hint : ?? pandas.DataFreame.to_csv)

In [ ]:
# ???????
import os
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

### ?????????

In [ ]:
# ?? data_path
dir_data = './data/'
f_app_train = os.path.join(dir_data, 'application_train.csv')
f_app_test = os.path.join(dir_data, 'application_test.csv')

# ????
app_train = pd.read_csv(f_app_train)
app_test = pd.read_csv(f_app_test)

In [ ]:
# ?? 2 ?????????????? (Label Encoding)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le_count = 0

# ????? column
for col in app_train:
    if app_train[col].dtype == 'object':
        # ?????????????
        if len(list(app_train[col].unique())) <= 2:
            # ?? Label Encoder
            le.fit(app_train[col])
            app_train[col] = le.transform(app_train[col])
            app_test[col] = le.transform(app_test[col])
            
            # ?????? columns ??????
            le_count += 1
            
# ???? (2???) ??? One Hot Encoding            
app_train = pd.get_dummies(app_train)
app_test = pd.get_dummies(app_test)

In [ ]:
# ???????????, ?????????, ??????????? (np.nan)
app_train['DAYS_EMPLOYED_ANOM'] = app_train["DAYS_EMPLOYED"] == 365243
app_train['DAYS_EMPLOYED'].replace({365243: np.nan}, inplace = True)
app_test['DAYS_EMPLOYED_ANOM'] = app_test["DAYS_EMPLOYED"] == 365243
app_test["DAYS_EMPLOYED"].replace({365243: np.nan}, inplace = True)

# ???? (DAYS_BIRTH) ???? 
app_train['DAYS_BIRTH'] = abs(app_train['DAYS_BIRTH'])
app_test['DAYS_BIRTH'] = abs(app_test['DAYS_BIRTH'])

### ?????
????????,????? training & testing data ???????,????? One hot encoding ???????,??????? training data ????? testing data ?,??????????????

In [ ]:
train_labels = app_train['TARGET']

# ?????, ????? training data ????? testing data ????
app_train, app_test = app_train.align(app_test, join = 'inner', axis = 1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler, Imputer

# ??????
train = app_train
features = list(train.columns)

# ?? test ??
test = app_test.copy()

# ??? : ?????????
imputer = Imputer(strategy = 'median')

# ??? : ??????? 0~1 ??
scaler = MinMaxScaler(feature_range = (0, 1))

# ??????????
imputer.fit(train)

# ?????? train, test ???????
train = imputer.transform(train)
test = imputer.transform(app_test)

# ????? train ????, ? train, test ??????
scaler.fit(train)
train = scaler.transform(train)
test = scaler.transform(test)

print('Training data shape: ', train.shape)
print('Testing data shape: ', test.shape)

### Fit the model

In [ ]:
from sklearn.linear_model import LogisticRegression

# ?????????
log_reg = LogisticRegression(C = 0.0001)

# ?? Train ??????
log_reg.fit(train, train_labels)

?? fit ???,??????? testing data ??????????????! (???? predict_proba ??????)

In [ ]:
# ???????
# ????????????? (??? 0 ???, ? 1 ???), ???????? 1 ?????
log_reg_pred = log_reg.predict_proba(test)[:, 1]

### ??????

In [ ]:
# ??????
submit = app_test[['SK_ID_CURR']]
submit['TARGET'] = log_reg_pred

submit.head()

In [ ]:
submit.to_csv("Day_016_HW.csv", index=False)